# 02 - Feature Engineering

## CyberForge AI - Security-Focused Feature Extraction

This notebook performs feature engineering for cybersecurity ML models.

### Feature Categories:
1. **URL Features** - Domain, path, query analysis
2. **Network Features** - Request patterns, headers, protocols
3. **JavaScript Behavior** - Script patterns, suspicious calls
4. **Browser Artifacts** - Cookies, localStorage, fingerprinting
5. **Security Indicators** - SSL, headers, CSP

### Alignment with Backend:
- Features match WebScraperAPIService output format
- Compatible with ThreatService detection patterns
- Supports real-time inference requirements

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List, Any, Optional
from urllib.parse import urlparse, parse_qs
import re
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load configuration
config_path = Path("../notebook_config.json")
with open(config_path) as f:
    CONFIG = json.load(f)

DATASETS_DIR = Path(CONFIG["datasets_dir"])
PROCESSED_DIR = DATASETS_DIR / "processed"
FEATURES_DIR = DATASETS_DIR / "features"
FEATURES_DIR.mkdir(exist_ok=True)

print(f"✓ Configuration loaded")
print(f"✓ Features output: {FEATURES_DIR}")

## 1. URL Feature Extraction

Extract security-relevant features from URLs.

In [ ]:
try:
    import tldextract
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'tldextract', '-q'])
    import tldextract

class URLFeatureExtractor:
    """
    Extract security-relevant features from URLs.
    Aligned with backend ThreatService URL analysis.
    """
    
    # Suspicious patterns from ThreatService
    SUSPICIOUS_KEYWORDS = ['phishing', 'malware', 'suspicious', 'hack', 'scam', 
                          'login', 'verify', 'account', 'secure', 'update']
    INJECTION_PATTERNS = [r'data:text/html', r'javascript:', r'vbscript:']
    
    def __init__(self):
        self.ip_pattern = re.compile(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}')
    
    def extract(self, url: str) -> Dict[str, Any]:
        """Extract all URL features"""
        if not isinstance(url, str) or not url:
            return self._empty_features()
        
        try:
            parsed = urlparse(url)
            extracted = tldextract.extract(url)
            
            features = {
                # Basic URL structure
                'url_length': len(url),
                'domain_length': len(parsed.netloc),
                'path_length': len(parsed.path),
                'query_length': len(parsed.query),
                
                # Domain analysis
                'subdomain_count': len(extracted.subdomain.split('.')) if extracted.subdomain else 0,
                'domain_depth': url.count('/') - 2,  # Minus protocol slashes
                'has_subdomain': len(extracted.subdomain) > 0,
                
                # Protocol security
                'is_https': parsed.scheme == 'https',
                'has_port': parsed.port is not None,
                'non_standard_port': parsed.port not in [None, 80, 443],
                
                # Suspicious indicators
                'has_ip_address': bool(self.ip_pattern.search(url)),
                'suspicious_keyword_count': sum(1 for kw in self.SUSPICIOUS_KEYWORDS if kw in url.lower()),
                'has_injection_pattern': any(re.search(p, url, re.I) for p in self.INJECTION_PATTERNS),
                
                # Character analysis
                'digit_count': sum(c.isdigit() for c in url),
                'special_char_count': sum(not c.isalnum() and c not in '/:.' for c in url),
                'hyphen_count': url.count('-'),
                'underscore_count': url.count('_'),
                'at_symbol': '@' in url,
                
                # Query parameters
                'param_count': len(parse_qs(parsed.query)),
                'has_query': len(parsed.query) > 0,
                
                # TLD analysis
                'tld': extracted.suffix,
                'tld_length': len(extracted.suffix),
                'is_common_tld': extracted.suffix in ['com', 'org', 'net', 'edu', 'gov'],
            }
            
            return features
            
        except Exception as e:
            return self._empty_features()
    
    def _empty_features(self) -> Dict:
        """Return empty feature dict for invalid URLs"""
        return {
            'url_length': 0, 'domain_length': 0, 'path_length': 0, 'query_length': 0,
            'subdomain_count': 0, 'domain_depth': 0, 'has_subdomain': False,
            'is_https': False, 'has_port': False, 'non_standard_port': False,
            'has_ip_address': False, 'suspicious_keyword_count': 0, 'has_injection_pattern': False,
            'digit_count': 0, 'special_char_count': 0, 'hyphen_count': 0, 'underscore_count': 0,
            'at_symbol': False, 'param_count': 0, 'has_query': False,
            'tld': '', 'tld_length': 0, 'is_common_tld': False
        }
    
    def extract_batch(self, urls: List[str]) -> pd.DataFrame:
        """Extract features from multiple URLs"""
        features = [self.extract(url) for url in urls]
        return pd.DataFrame(features)

url_extractor = URLFeatureExtractor()
print("✓ URL Feature Extractor initialized")

# Test
test_features = url_extractor.extract("https://suspicious-login.example.com/verify?id=123")
print(f"\nTest features extracted: {len(test_features)} features")

## 2. Network Request Feature Extraction

Features for HTTP request analysis (aligned with WebScraperAPIService).

In [ ]:
class NetworkFeatureExtractor:
    """
    Extract features from network request data.
    Matches WebScraperAPIService network_requests format.
    """
    
    RISKY_CONTENT_TYPES = ['application/javascript', 'text/javascript', 'application/x-javascript']
    
    def extract_from_requests(self, requests: List[Dict]) -> Dict[str, Any]:
        """Extract features from a list of network requests"""
        if not requests:
            return self._empty_features()
        
        # Request type counts
        types = [r.get('type', 'unknown').lower() for r in requests]
        methods = [r.get('method', 'GET').upper() for r in requests]
        statuses = [r.get('status', 0) for r in requests]
        
        return {
            # Volume metrics
            'total_requests': len(requests),
            'script_requests': types.count('script'),
            'xhr_requests': types.count('xhr'),
            'image_requests': types.count('image'),
            'stylesheet_requests': types.count('stylesheet'),
            'document_requests': types.count('document'),
            
            # Method distribution
            'get_requests': methods.count('GET'),
            'post_requests': methods.count('POST'),
            'other_method_requests': len([m for m in methods if m not in ['GET', 'POST']]),
            
            # Status analysis
            'successful_requests': sum(1 for s in statuses if 200 <= s < 300),
            'redirect_requests': sum(1 for s in statuses if 300 <= s < 400),
            'client_error_requests': sum(1 for s in statuses if 400 <= s < 500),
            'server_error_requests': sum(1 for s in statuses if s >= 500),
            'failed_request_ratio': sum(1 for s in statuses if s >= 400) / max(len(requests), 1),
            
            # Size metrics
            'total_size_kb': sum(r.get('size', 0) for r in requests) / 1024,
            'avg_request_size': np.mean([r.get('size', 0) for r in requests]) if requests else 0,
            
            # Domain diversity
            'unique_domains': len(set(self._extract_domain(r.get('url', '')) for r in requests)),
            'third_party_ratio': self._calculate_third_party_ratio(requests),
        }
    
    def _extract_domain(self, url: str) -> str:
        try:
            return urlparse(url).netloc
        except:
            return ''
    
    def _calculate_third_party_ratio(self, requests: List[Dict]) -> float:
        if not requests:
            return 0.0
        domains = [self._extract_domain(r.get('url', '')) for r in requests]
        if not domains:
            return 0.0
        main_domain = max(set(domains), key=domains.count) if domains else ''
        third_party = sum(1 for d in domains if d and d != main_domain)
        return third_party / len(requests)
    
    def _empty_features(self) -> Dict:
        return {
            'total_requests': 0, 'script_requests': 0, 'xhr_requests': 0,
            'image_requests': 0, 'stylesheet_requests': 0, 'document_requests': 0,
            'get_requests': 0, 'post_requests': 0, 'other_method_requests': 0,
            'successful_requests': 0, 'redirect_requests': 0,
            'client_error_requests': 0, 'server_error_requests': 0, 'failed_request_ratio': 0,
            'total_size_kb': 0, 'avg_request_size': 0,
            'unique_domains': 0, 'third_party_ratio': 0
        }

network_extractor = NetworkFeatureExtractor()
print("✓ Network Feature Extractor initialized")

## 3. Security Header Feature Extraction

Features based on HTTP security headers.

In [ ]:
class SecurityHeaderExtractor:
    """
    Extract features from HTTP security headers.
    Aligned with WebScraperAPIService security_report.
    """
    
    SECURITY_HEADERS = [
        'Content-Security-Policy',
        'X-Content-Type-Options',
        'X-Frame-Options',
        'X-XSS-Protection',
        'Strict-Transport-Security',
        'Referrer-Policy',
        'Permissions-Policy',
        'X-Permitted-Cross-Domain-Policies'
    ]
    
    def extract(self, headers: Dict[str, str], security_report: Dict = None) -> Dict[str, Any]:
        """Extract security header features"""
        headers_lower = {k.lower(): v for k, v in (headers or {}).items()}
        
        features = {}
        
        # Check each security header
        for header in self.SECURITY_HEADERS:
            key = f"has_{header.lower().replace('-', '_')}"
            features[key] = header.lower() in headers_lower
        
        # Aggregate metrics
        features['security_headers_count'] = sum(1 for h in self.SECURITY_HEADERS if h.lower() in headers_lower)
        features['security_headers_ratio'] = features['security_headers_count'] / len(self.SECURITY_HEADERS)
        features['missing_security_headers'] = len(self.SECURITY_HEADERS) - features['security_headers_count']
        
        # From security report if available
        if security_report:
            features['is_https'] = security_report.get('is_https', False)
            features['has_mixed_content'] = security_report.get('mixed_content', False)
            features['has_insecure_cookies'] = security_report.get('insecure_cookies', False)
        
        return features
    
    def calculate_security_score(self, features: Dict) -> float:
        """Calculate overall security score (0-100)"""
        score = 0
        
        # Headers (40 points max)
        score += features.get('security_headers_ratio', 0) * 40
        
        # HTTPS (30 points)
        if features.get('is_https', False):
            score += 30
        
        # No mixed content (15 points)
        if not features.get('has_mixed_content', True):
            score += 15
        
        # Secure cookies (15 points)
        if not features.get('has_insecure_cookies', True):
            score += 15
        
        return min(100, max(0, score))

header_extractor = SecurityHeaderExtractor()
print("✓ Security Header Extractor initialized")

## 4. JavaScript Behavior Feature Extraction

In [ ]:
class JavaScriptFeatureExtractor:
    """
    Extract features from JavaScript behavior analysis.
    Supports desktop app browser monitoring.
    """
    
    SUSPICIOUS_APIS = [
        'eval', 'document.write', 'innerHTML', 'outerHTML',
        'localStorage', 'sessionStorage', 'indexedDB',
        'navigator.geolocation', 'navigator.credentials',
        'crypto.subtle', 'WebSocket'
    ]
    
    OBFUSCATION_PATTERNS = [
        r'\\x[0-9a-fA-F]{2}',  # Hex encoding
        r'\\u[0-9a-fA-F]{4}',  # Unicode encoding
        r'atob\(',              # Base64 decode
        r'String\.fromCharCode', # Char code obfuscation
        r'unescape\(',          # URL decode
    ]
    
    def extract_from_console_logs(self, logs: List[Dict]) -> Dict[str, Any]:
        """Extract features from console logs"""
        if not logs:
            return self._empty_features()
        
        levels = [log.get('level', 'log').lower() for log in logs]
        messages = [log.get('message', '') for log in logs]
        all_text = ' '.join(messages)
        
        return {
            'console_log_count': len(logs),
            'console_error_count': levels.count('error'),
            'console_warning_count': levels.count('warning'),
            'console_info_count': levels.count('info'),
            'error_ratio': levels.count('error') / max(len(logs), 1),
            'has_security_errors': any('security' in m.lower() or 'cors' in m.lower() for m in messages),
            'has_csp_violations': any('content security policy' in m.lower() for m in messages),
        }
    
    def analyze_script_content(self, script: str) -> Dict[str, Any]:
        """Analyze JavaScript code for suspicious patterns"""
        if not script:
            return self._empty_script_features()
        
        return {
            'script_length': len(script),
            'suspicious_api_count': sum(1 for api in self.SUSPICIOUS_APIS if api in script),
            'obfuscation_score': sum(len(re.findall(p, script)) for p in self.OBFUSCATION_PATTERNS),
            'has_eval': 'eval(' in script or 'eval (' in script,
            'has_document_write': 'document.write' in script,
            'has_inline_event_handlers': bool(re.search(r'on\w+\s*=', script)),
            'external_url_count': len(re.findall(r'https?://[^\s"\')]+', script)),
            'function_count': len(re.findall(r'function\s*\w*\s*\(', script)),
        }
    
    def _empty_features(self) -> Dict:
        return {
            'console_log_count': 0, 'console_error_count': 0, 'console_warning_count': 0,
            'console_info_count': 0, 'error_ratio': 0, 'has_security_errors': False,
            'has_csp_violations': False
        }
    
    def _empty_script_features(self) -> Dict:
        return {
            'script_length': 0, 'suspicious_api_count': 0, 'obfuscation_score': 0,
            'has_eval': False, 'has_document_write': False, 'has_inline_event_handlers': False,
            'external_url_count': 0, 'function_count': 0
        }

js_extractor = JavaScriptFeatureExtractor()
print("✓ JavaScript Feature Extractor initialized")

## 5. Unified Feature Pipeline

In [ ]:
class CyberForgeFeaturePipeline:
    """
    Unified feature extraction pipeline for CyberForge AI.
    Combines all extractors for comprehensive security feature engineering.
    """
    
    def __init__(self):
        self.url_extractor = URLFeatureExtractor()
        self.network_extractor = NetworkFeatureExtractor()
        self.header_extractor = SecurityHeaderExtractor()
        self.js_extractor = JavaScriptFeatureExtractor()
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()
        self.feature_names = []
    
    def extract_website_features(self, scraped_data: Dict) -> Dict[str, Any]:
        """Extract all features from website scraped data"""
        features = {}
        
        # URL features
        url_features = self.url_extractor.extract(scraped_data.get('url', ''))
        features.update({f"url_{k}": v for k, v in url_features.items() if k != 'tld'})
        
        # Network features
        network_features = self.network_extractor.extract_from_requests(
            scraped_data.get('network_requests', [])
        )
        features.update({f"net_{k}": v for k, v in network_features.items()})
        
        # Security header features
        header_features = self.header_extractor.extract(
            scraped_data.get('response_headers', {}),
            scraped_data.get('security_report', {})
        )
        features.update({f"sec_{k}": v for k, v in header_features.items()})
        
        # JavaScript features
        js_features = self.js_extractor.extract_from_console_logs(
            scraped_data.get('console_logs', [])
        )
        features.update({f"js_{k}": v for k, v in js_features.items()})
        
        # Calculate risk score
        features['security_score'] = self.header_extractor.calculate_security_score(header_features)
        
        return features
    
    def process_dataset(self, df: pd.DataFrame, url_column: str = 'url') -> pd.DataFrame:
        """Process a dataset and extract URL features"""
        if url_column not in df.columns:
            print(f"  ⚠ No '{url_column}' column found")
            return df
        
        # Extract URL features
        url_features = df[url_column].apply(lambda x: self.url_extractor.extract(x))
        url_df = pd.DataFrame(url_features.tolist())
        url_df.columns = [f"url_{c}" for c in url_df.columns if c != 'tld']
        
        # Combine with original features
        result = pd.concat([df.reset_index(drop=True), url_df.reset_index(drop=True)], axis=1)
        
        return result
    
    def prepare_for_training(self, df: pd.DataFrame, label_column: str = 'label') -> tuple:
        """Prepare features for model training"""
        df = df.copy()
        
        # Separate features and labels
        if label_column in df.columns:
            y = df[label_column]
            X = df.drop(columns=[label_column])
        else:
            y = None
            X = df
        
        # Select numeric columns only
        numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        X_numeric = X[numeric_cols].fillna(0)
        
        # Convert boolean to int
        bool_cols = X.select_dtypes(include=[bool]).columns.tolist()
        for col in bool_cols:
            X_numeric[col] = X[col].astype(int)
        
        self.feature_names = X_numeric.columns.tolist()
        
        # Encode labels if present
        if y is not None:
            if y.dtype == 'object':
                y = self.label_encoder.fit_transform(y)
            else:
                y = y.values
        
        return X_numeric, y

pipeline = CyberForgeFeaturePipeline()
print("✓ Feature Pipeline initialized")

## 6. Process Datasets

In [ ]:
# Load manifest
manifest_path = PROCESSED_DIR / "manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"✓ Loaded manifest with {len(manifest)} datasets")
else:
    print("⚠ No manifest found. Run 01_data_acquisition.ipynb first.")
    manifest = []

In [ ]:
# Process each dataset
processed_datasets = {}
feature_stats = []

print("Processing datasets for feature engineering...\n")

for entry in manifest:
    name = entry['name']
    path = Path("..") / entry['path']
    
    if not path.exists():
        print(f"  ⚠ {name}: File not found")
        continue
    
    print(f"  Processing: {name}")
    
    try:
        df = pd.read_csv(path)
        
        # Check for URL column to extract URL features
        url_cols = [c for c in df.columns if 'url' in c.lower()]
        if url_cols:
            df = pipeline.process_dataset(df, url_column=url_cols[0])
        
        # Prepare for training
        X, y = pipeline.prepare_for_training(df)
        
        processed_datasets[name] = {
            'X': X,
            'y': y,
            'feature_names': pipeline.feature_names,
            'n_samples': len(X),
            'n_features': len(pipeline.feature_names)
        }
        
        print(f"    ✓ {len(X)} samples, {len(pipeline.feature_names)} features")
        
        feature_stats.append({
            'name': name,
            'samples': len(X),
            'features': len(pipeline.feature_names),
            'has_labels': y is not None
        })
        
    except Exception as e:
        print(f"    ⚠ Error: {e}")

print(f"\n✓ Processed {len(processed_datasets)} datasets")

## 7. Save Feature-Engineered Data

In [ ]:
import joblib

# Save processed datasets
feature_manifest = []

print("Saving feature-engineered datasets...")

for name, data in processed_datasets.items():
    # Save as parquet for efficiency
    output_path = FEATURES_DIR / f"{name}_features.parquet"
    
    # Create dataframe with features
    df_features = data['X'].copy()
    if data['y'] is not None:
        df_features['label'] = data['y']
    
    df_features.to_parquet(output_path, index=False)
    
    feature_manifest.append({
        'name': name,
        'path': str(output_path.relative_to(DATASETS_DIR.parent)),
        'samples': data['n_samples'],
        'features': data['n_features'],
        'feature_names': data['feature_names'],
        'has_labels': data['y'] is not None
    })
    
    print(f"  ✓ Saved: {output_path.name}")

# Save feature manifest
manifest_path = FEATURES_DIR / "feature_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(feature_manifest, f, indent=2)

# Save pipeline for inference
pipeline_path = FEATURES_DIR / "feature_pipeline.pkl"
joblib.dump(pipeline, pipeline_path)

print(f"\n✓ Feature manifest saved to: {manifest_path}")
print(f"✓ Feature pipeline saved to: {pipeline_path}")

## 8. Summary

In [ ]:
print("\n" + "=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)

total_samples = sum(d['n_samples'] for d in processed_datasets.values())
total_features = max(d['n_features'] for d in processed_datasets.values()) if processed_datasets else 0

print(f"""
🔧 Feature Engineering Summary:
   - Datasets processed: {len(processed_datasets)}
   - Total samples: {total_samples:,}
   - Max features: {total_features}
   - Output directory: {FEATURES_DIR}

📊 Feature Categories:
   - URL Features: Domain, path, security indicators
   - Network Features: Request patterns, status codes
   - Security Headers: CSP, HSTS, X-Frame-Options
   - JavaScript: Console logs, suspicious APIs

📁 Datasets Ready for Training:""")

for entry in feature_manifest:
    print(f"   ✓ {entry['name']}: {entry['samples']:,} samples, {entry['features']} features")

print(f"""
Next step:
  → 03_model_training.ipynb
""")
print("=" * 60)